# 05 - Imageamento por Microscopia Eletronica
> Processamento de imagem, analise de falhas e automacao com Python

**Modulo:** `13_enfitec_projetos` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 11
print('Bibliotecas carregadas!')

## Gerando Imagens Sinteticas de MEV

Vamos criar imagens sinteticas que simulam microestruturas tipicas
observadas em microscopia eletronica de varredura (MEV/SEM):
- Graos com diferentes niveis de cinza
- Contornos de grao
- Porosidade (poros escuros)


In [ ]:
def generate_grain_structure(size=512, n_seeds=80, n_pores=25, rng_seed=42):
    """Gera imagem sintetica de microestrutura com graos e poros."""
    rng = np.random.RandomState(rng_seed)
    
    # Gerar sementes de Voronoi para graos
    seeds_x = rng.randint(0, size, n_seeds)
    seeds_y = rng.randint(0, size, n_seeds)
    grain_values = rng.randint(80, 220, n_seeds)  # Niveis de cinza
    
    # Criar mapa de graos via Voronoi (distancia mais proxima)
    y_grid, x_grid = np.mgrid[0:size, 0:size]
    grain_map = np.zeros((size, size), dtype=np.float64)
    label_map = np.zeros((size, size), dtype=np.int32)
    
    # Calcular distancia a cada semente
    min_dist = np.full((size, size), np.inf)
    for i in range(n_seeds):
        dist = np.sqrt((x_grid - seeds_x[i])**2 + (y_grid - seeds_y[i])**2)
        mask = dist < min_dist
        min_dist[mask] = dist[mask]
        grain_map[mask] = grain_values[i]
        label_map[mask] = i + 1
    
    # Detectar contornos de grao (gradiente do label_map)
    gx = np.diff(label_map, axis=1, prepend=label_map[:, :1])
    gy = np.diff(label_map, axis=0, prepend=label_map[:1, :])
    boundaries = (np.abs(gx) + np.abs(gy)) > 0
    
    # Contornos escuros
    grain_map[boundaries] = grain_map[boundaries] * 0.3
    
    # Adicionar poros (circulos escuros)
    pore_mask = np.zeros((size, size), dtype=bool)
    pore_radii = []
    for _ in range(n_pores):
        px, py = rng.randint(20, size-20), rng.randint(20, size-20)
        radius = rng.randint(3, 15)
        pore_radii.append(radius)
        dist = np.sqrt((x_grid - px)**2 + (y_grid - py)**2)
        pore = dist < radius
        pore_mask |= pore
    
    grain_map[pore_mask] = rng.uniform(10, 40, np.sum(pore_mask))
    
    # Adicionar ruido gaussiano (ruido de detector)
    noise = rng.normal(0, 5, (size, size))
    image = np.clip(grain_map + noise, 0, 255).astype(np.uint8)
    
    return image, label_map, boundaries, pore_mask

image, labels, boundaries, pores = generate_grain_structure()
print(f'Imagem gerada: {image.shape}')
print(f'Numero de graos: {len(np.unique(labels)) - 1}')
print(f'Faixa de valores: [{image.min()}, {image.max()}]')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(image, cmap='gray')
axes[0].set_title('Imagem MEV Sintetica')
axes[0].axis('off')

axes[1].imshow(labels, cmap='nipy_spectral')
axes[1].set_title('Mapa de Graos (Voronoi)')
axes[1].axis('off')

# Sobrepor contornos e poros
overlay = np.stack([image]*3, axis=-1).astype(float) / 255
overlay[boundaries, 0] = 1.0
overlay[boundaries, 1] = 0.2
overlay[boundaries, 2] = 0.2
overlay[pores, 0] = 0.2
overlay[pores, 1] = 0.2
overlay[pores, 2] = 1.0
axes[2].imshow(overlay)
axes[2].set_title('Contornos (vermelho) + Poros (azul)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## Pre-processamento de Imagens

Vamos pedir ao Claude para gerar codigo Python de pre-processamento
tipico para imagens de MEV: melhoria de contraste, reducao de ruido
e nitidez.


In [ ]:
preproc_prompt = """Sou engenheiro de materiais trabalhando com imagens de MEV
(microscopia eletronica de varredura) de uma microestrutura metalica.
A imagem eh uma matriz numpy 2D (512x512, uint8, escala de cinza).

Gere codigo Python COMPLETO (usando apenas numpy e scipy, sem OpenCV)
para as seguintes etapas de pre-processamento:

1. Equalizacao de histograma para melhorar contraste
2. Filtro gaussiano para reducao de ruido (sigma=1.5)
3. Mascara unsharp para realce de bordas
4. Normalizacao para faixa 0-255

Cada funcao deve receber e retornar um array numpy 2D.
Use docstrings simples. Nao use acentos."""

resp_preproc = ask(preproc_prompt, model=SONNET, max_tokens=2000)
print(resp_preproc)

In [ ]:
from scipy.ndimage import gaussian_filter

def equalize_histogram(img):
    """Equalizacao de histograma para melhoria de contraste."""
    hist, bins = np.histogram(img.flatten(), 256, [0, 256])
    cdf = hist.cumsum()
    cdf_normalized = (cdf - cdf.min()) * 255 / (cdf.max() - cdf.min())
    return cdf_normalized[img.astype(int)].astype(np.uint8)

def denoise_gaussian(img, sigma=1.5):
    """Reducao de ruido com filtro gaussiano."""
    return gaussian_filter(img.astype(float), sigma=sigma)

def unsharp_mask(img, sigma=2.0, strength=1.5):
    """Realce de bordas com mascara unsharp."""
    blurred = gaussian_filter(img.astype(float), sigma=sigma)
    sharpened = img.astype(float) + strength * (img.astype(float) - blurred)
    return np.clip(sharpened, 0, 255)

# Aplicar pipeline
img_eq = equalize_histogram(image)
img_denoised = denoise_gaussian(img_eq, sigma=1.5)
img_sharp = unsharp_mask(img_denoised, sigma=2.0, strength=1.5)
img_final = np.clip(img_sharp, 0, 255).astype(np.uint8)

print(f'Original  - media: {image.mean():.1f}, std: {image.std():.1f}')
print(f'Processada - media: {img_final.mean():.1f}, std: {img_final.std():.1f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

axes[0, 0].imshow(image, cmap='gray')
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

axes[0, 1].imshow(img_eq, cmap='gray')
axes[0, 1].set_title('Apos Equalizacao')
axes[0, 1].axis('off')

axes[1, 0].imshow(img_denoised, cmap='gray')
axes[1, 0].set_title('Apos Filtro Gaussiano')
axes[1, 0].axis('off')

axes[1, 1].imshow(img_final, cmap='gray')
axes[1, 1].set_title('Apos Unsharp Mask (Final)')
axes[1, 1].axis('off')

plt.suptitle('Pipeline de Pre-processamento MEV', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Segmentacao e Analise de Graos

Vamos implementar deteccao de contornos de grao e calcular a
distribuicao de tamanho de graos.


In [ ]:
grain_prompt = """Tenho uma imagem de MEV (512x512, escala de cinza, numpy array)
de uma microestrutura com graos de diferentes tons de cinza e contornos
escuros entre eles. Preciso:

1. Estrategia para segmentacao dos graos usando apenas numpy e scipy
   (sem OpenCV/skimage)
2. Como calcular o tamanho de cada grao (area em pixels e diametro equivalente)
3. Como gerar a distribuicao de tamanho de graos
4. Qual metodo usar para calcular o tamanho medio (ASTM E112)

Explique a abordagem passo a passo. Seja conciso e tecnico."""

resp_grain = ask(grain_prompt, model=SONNET, max_tokens=1500)
print(resp_grain)

In [ ]:
from scipy.ndimage import label, find_objects
from scipy.ndimage import binary_dilation, binary_erosion

# Segmentar graos usando o label_map que ja temos
# (Em caso real, usariamos thresholding + watershed)
n_grains = len(np.unique(labels)) - 1  # Excluir fundo

# Calcular propriedades de cada grao
grain_areas = []
grain_diameters = []
grain_centroids = []

for i in range(1, n_grains + 1):
    mask = labels == i
    area = np.sum(mask)
    
    # Ignorar graos muito pequenos (artefatos) e bordas
    if area < 50:
        continue
    
    # Diametro equivalente (circulo de mesma area)
    d_eq = 2 * np.sqrt(area / np.pi)
    
    # Centroide
    ys, xs = np.where(mask)
    cx, cy = np.mean(xs), np.mean(ys)
    
    grain_areas.append(area)
    grain_diameters.append(d_eq)
    grain_centroids.append((cx, cy))

grain_areas = np.array(grain_areas)
grain_diameters = np.array(grain_diameters)

# Converter para micrometros (supondo escala: 1 pixel = 0.5 um)
SCALE = 0.5  # um/pixel
diameters_um = grain_diameters * SCALE
areas_um2 = grain_areas * SCALE**2

print(f'Graos detectados: {len(grain_areas)}')
print(f'Diametro medio: {np.mean(diameters_um):.1f} um')
print(f'Diametro mediano: {np.median(diameters_um):.1f} um')
print(f'Desvio padrao: {np.std(diameters_um):.1f} um')
print(f'Faixa: {np.min(diameters_um):.1f} - {np.max(diameters_um):.1f} um')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Mapa de graos colorido
axes[0].imshow(labels, cmap='nipy_spectral')
axes[0].set_title(f'Segmentacao ({len(grain_areas)} graos)')
axes[0].axis('off')

# Histograma de diametros
axes[1].hist(diameters_um, bins=20, color='steelblue', edgecolor='black', alpha=0.8)
axes[1].axvline(np.mean(diameters_um), color='red', linestyle='--',
                label=f'Media = {np.mean(diameters_um):.1f} um')
axes[1].axvline(np.median(diameters_um), color='orange', linestyle='--',
                label=f'Mediana = {np.median(diameters_um):.1f} um')
axes[1].set_xlabel('Diametro equivalente (um)')
axes[1].set_ylabel('Frequencia')
axes[1].set_title('Distribuicao de Tamanho de Graos')
axes[1].legend()

# Distribuicao cumulativa
sorted_d = np.sort(diameters_um)
cumulative = np.arange(1, len(sorted_d) + 1) / len(sorted_d) * 100
axes[2].plot(sorted_d, cumulative, 'b-', linewidth=2)
axes[2].axhline(50, color='gray', linestyle=':', alpha=0.5)
d50 = np.interp(50, cumulative, sorted_d)
axes[2].axvline(d50, color='red', linestyle='--', label=f'D50 = {d50:.1f} um')
axes[2].set_xlabel('Diametro equivalente (um)')
axes[2].set_ylabel('Cumulativo (%)')
axes[2].set_title('Distribuicao Cumulativa')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Analise de Porosidade

Vamos medir a porosidade da microestrutura: fracao de area,
distribuicao de tamanho de poros e circularidade.


In [ ]:
porosity_prompt = """Tenho uma imagem binaria de poros (True = poro) extraida
de uma imagem MEV de um material ceramico sinterizado.
A imagem tem 512x512 pixels com escala de 0.5 um/pixel.

Preciso calcular usando numpy e scipy (sem OpenCV):
1. Fracao de area porosa (porosidade percentual)
2. Numero e tamanho individual de cada poro (usando scipy.ndimage.label)
3. Circularidade de cada poro: C = 4*pi*Area / Perimetro^2
4. Classificacao: poro circular (C>0.8), alongado (0.5<C<0.8), irregular (C<0.5)

Explique como estimar o perimetro de cada poro a partir da mascara binaria.
Seja conciso e tecnico."""

resp_porosity = ask(porosity_prompt, model=SONNET, max_tokens=1500)
print(resp_porosity)

In [ ]:
from scipy.ndimage import label as nd_label

# Segmentar poros individuais
labeled_pores, n_pores_found = nd_label(pores)

# Porosidade total
total_area = image.shape[0] * image.shape[1]
pore_area_total = np.sum(pores)
porosity_pct = (pore_area_total / total_area) * 100

# Analisar cada poro
pore_data = []
for i in range(1, n_pores_found + 1):
    mask_i = labeled_pores == i
    area_px = np.sum(mask_i)
    
    if area_px < 5:  # Ignorar artefatos
        continue
    
    # Perimetro: contar pixels de borda
    eroded = binary_erosion(mask_i)
    perimeter_px = np.sum(mask_i) - np.sum(eroded)
    perimeter_px = max(perimeter_px, 1)  # Evitar divisao por zero
    
    # Circularidade
    circularity = 4 * np.pi * area_px / (perimeter_px ** 2)
    circularity = min(circularity, 1.0)  # Limitar a 1.0
    
    # Diametro equivalente
    d_eq = 2 * np.sqrt(area_px / np.pi) * SCALE
    
    # Classificacao
    if circularity > 0.8:
        shape = 'circular'
    elif circularity > 0.5:
        shape = 'alongado'
    else:
        shape = 'irregular'
    
    pore_data.append({
        'area_um2': area_px * SCALE**2,
        'diameter_um': d_eq,
        'circularity': circularity,
        'shape': shape
    })

print(f'Porosidade total: {porosity_pct:.2f}%')
print(f'Poros identificados: {len(pore_data)}')
print(f'\nDistribuicao de forma:')
shapes = [p['shape'] for p in pore_data]
for s in ['circular', 'alongado', 'irregular']:
    count = shapes.count(s)
    print(f'  {s}: {count} ({count/len(shapes)*100:.0f}%)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Mapa de poros
axes[0].imshow(image, cmap='gray')
pore_overlay = np.ma.masked_where(~pores, np.ones_like(image))
axes[0].imshow(pore_overlay, cmap='Reds', alpha=0.6)
axes[0].set_title(f'Poros Detectados (Porosidade: {porosity_pct:.2f}%)')
axes[0].axis('off')

# Distribuicao de tamanho de poros
pore_diams = [p['diameter_um'] for p in pore_data]
axes[1].hist(pore_diams, bins=15, color='coral', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Diametro equivalente (um)')
axes[1].set_ylabel('Frequencia')
axes[1].set_title('Distribuicao de Tamanho de Poros')
axes[1].axvline(np.mean(pore_diams), color='red', linestyle='--',
                label=f'Media = {np.mean(pore_diams):.1f} um')
axes[1].legend()

# Circularidade vs tamanho
circs = [p['circularity'] for p in pore_data]
colors_map = {'circular': 'green', 'alongado': 'orange', 'irregular': 'red'}
for p in pore_data:
    axes[2].scatter(p['diameter_um'], p['circularity'],
                    c=colors_map[p['shape']], s=40, alpha=0.7, edgecolors='black', linewidth=0.5)
axes[2].axhline(0.8, color='green', linestyle=':', alpha=0.5, label='Circular > 0.8')
axes[2].axhline(0.5, color='orange', linestyle=':', alpha=0.5, label='Alongado > 0.5')
axes[2].set_xlabel('Diametro equivalente (um)')
axes[2].set_ylabel('Circularidade')
axes[2].set_title('Circularidade vs Tamanho')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Analise de Falhas

Vamos descrever caracteristicas de uma superficie de fratura e pedir
ao Claude para identificar o modo de falha.


In [ ]:
# Gerar imagem sintetica de superficie de fratura
np.random.seed(77)
size = 512

# Base: textura de fratura (ruido + padroes direcionais)
fracture = np.random.normal(128, 30, (size, size))

# Adicionar marcas de rio (river marks) - tipico de fratura fragil
y, x = np.mgrid[0:size, 0:size]
for i in range(8):
    angle = np.random.uniform(0.3, 0.7) * np.pi
    offset = np.random.uniform(-100, 100)
    width = np.random.uniform(1.5, 3.0)
    line = np.abs(y * np.cos(angle) - x * np.sin(angle) + offset)
    mask = line < width
    fracture[mask] = np.random.uniform(40, 70)

# Adicionar facetas de clivagem (regioes planas)
for _ in range(12):
    cx, cy = np.random.randint(50, size-50, 2)
    rx, ry = np.random.randint(20, 60, 2)
    level = np.random.uniform(100, 200)
    ellipse = ((x - cx)**2 / rx**2 + (y - cy)**2 / ry**2) < 1
    fracture[ellipse] = level + np.random.normal(0, 3, np.sum(ellipse))

# Suavizar levemente
fracture = gaussian_filter(fracture, sigma=0.8)
fracture_img = np.clip(fracture, 0, 255).astype(np.uint8)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(fracture_img, cmap='gray')
ax.set_title('Superficie de Fratura - Imagem MEV Sintetica', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Calcular metricas da superficie de fratura
from scipy.ndimage import sobel

# Rugosidade (desvio padrao local)
roughness = np.std(fracture_img.astype(float))

# Deteccao de bordas (intensidade de gradiente)
edge_x = sobel(fracture_img.astype(float), axis=0)
edge_y = sobel(fracture_img.astype(float), axis=1)
edge_mag = np.sqrt(edge_x**2 + edge_y**2)

# Orientacao preferencial
edge_angle = np.arctan2(edge_y, edge_x) * 180 / np.pi
strong_edges = edge_mag > np.percentile(edge_mag, 90)

# Histograma de niveis de cinza
hist_values, bin_edges = np.histogram(fracture_img.flatten(), bins=50)
# Bimodalidade (indicaria duas fases)
hist_norm = hist_values / hist_values.max()

frac_description = f"""Analisei uma imagem MEV (512x512 pixels, 0.5 um/pixel) de uma
superficie de fratura de um aco baixo carbono (AISI 1020). As metricas
calculadas por processamento de imagem sao:

Metricas da imagem:
- Rugosidade (desvio padrao de cinza): {roughness:.1f}
- Intensidade media de gradiente: {np.mean(edge_mag):.1f}
- Porcentagem de bordas fortes (>P90): {np.sum(strong_edges)/strong_edges.size*100:.1f}%
- Nivel de cinza medio: {np.mean(fracture_img):.1f}
- Faixa de niveis: {np.min(fracture_img)} - {np.max(fracture_img)}

Descricao visual da superficie:
- Presenca de linhas retas escuras convergindo para uma regiao (river marks)
- Areas planas com nivel de cinza uniforme (facetas de clivagem)
- Ausencia de dimples (microcavidades) ou estricao
- Textura geral relativamente plana e angulosa
- Material: aco AISI 1020, condicao de servico a -20 graus Celsius

Por favor:
1. Identifique o modo de falha predominante (ductil, fragil, fadiga, CST)
2. Justifique com base nas caracteristicas observadas
3. Explique o mecanismo microestrutural provavel
4. Sugira a causa raiz e acoes corretivas
5. Indique ensaios complementares recomendados

Responda como perito em analise de falhas."""

resp_failure = ask(frac_description, model=SONNET, max_tokens=2000)
print(resp_failure)

## Relatorio de Microscopia Automatizado

Vamos gerar um relatorio completo consolidando todas as analises
realizadas neste notebook.


In [ ]:
report_prompt = f"""Gere um relatorio tecnico completo de analise por microscopia
eletronica de varredura (MEV) com os seguintes resultados:

EQUIPAMENTO: MEV com detector de eletrons secundarios (SE)
ESCALA: 0.5 um/pixel
RESOLUCAO: 512 x 512 pixels

1. ANALISE MICROESTRUTURAL:
   - Graos identificados: {len(grain_areas)}
   - Diametro medio: {np.mean(diameters_um):.1f} um
   - Diametro mediano (D50): {d50:.1f} um
   - Desvio padrao: {np.std(diameters_um):.1f} um
   - Faixa: {np.min(diameters_um):.1f} - {np.max(diameters_um):.1f} um

2. ANALISE DE POROSIDADE:
   - Porosidade total: {porosity_pct:.2f}%
   - Numero de poros: {len(pore_data)}
   - Diametro medio de poro: {np.mean(pore_diams):.1f} um
   - Poros circulares: {shapes.count('circular')}
   - Poros alongados: {shapes.count('alongado')}
   - Poros irregulares: {shapes.count('irregular')}

3. ANALISE DE FRATURA (amostra separada - aco AISI 1020):
   - Rugosidade superficial: {roughness:.1f}
   - Caracteristicas: river marks, facetas de clivagem
   - Sem dimples ou estricao
   - Condicao: servico a -20 graus Celsius

O relatorio deve conter:
1. Objetivo e escopo
2. Metodologia (aquisicao e processamento digital)
3. Resultados: microestrutura, porosidade, fractografia
4. Discussao e interpretacao
5. Conclusoes e recomendacoes

Use formato de laudo tecnico profissional."""

report = ask(report_prompt, model=SONNET, max_tokens=3000)
print(report)

## Exercicios

1. **Deteccao de inclusoes**: Modifique a funcao de geracao de imagem para
   adicionar inclusoes (particulas brilhantes dentro dos graos). Implemente
   deteccao por thresholding e classifique por tamanho.

2. **Analise de textura**: Calcule a matriz de co-ocorrencia (GLCM) da
   imagem usando numpy e extraia parametros de Haralick (contraste,
   homogeneidade, energia). Peca ao Claude para interpretar.

3. **Mapeamento EDS**: Gere mapas sinteticos de composicao quimica
   (Fe, Cr, Ni) associados aos graos e peca ao Claude para identificar
   fases presentes (austenita, ferrita, sigma).

4. **Estereologia quantitativa**: Implemente o metodo de intercepto linear
   (Heyn) para medir tamanho de grao conforme ASTM E112. Compare com
   o metodo de area equivalente.

5. **Analise de fratura ductil**: Gere uma imagem sintetica com dimples
   (microcavidades hemisfericas) e peca ao Claude para contrastar com
   a fratura fragil analisada neste notebook.


## Proximos Passos

- Integrar com bibliotecas especializadas (scikit-image, OpenCV)
- Estudar tecnicas de deep learning para segmentacao de microestruturas
- Implementar pipeline automatizado de analise batch (multiplas imagens)
- Explorar reconstrucao 3D a partir de series de imagens FIB-SEM
- Correlacionar microestrutura com propriedades mecanicas

---
*Notebook gerado como material didatico do modulo 13_enfitec_projetos.*
